# Titanic competition
> Author | Lavrov Evgeniy

## 1. Import Libraries & Load Data

In [1]:
import kagglehub
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

/home/koting/_code/Kaggle-Solutions/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
path = kagglehub.competition_download("titanic")
train_df = pd.read_csv(f"{path}/train.csv")
train_df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
test_df = pd.read_csv(f"{path}/test.csv")
test_df.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


## 2. Processing dataframes

In [4]:
train_df = train_df.copy()

train_df['Title'] = train_df['Name'].str.extract(r',\s*([A-Za-z]+)\.', expand=False)
rare_titles = ['Dr', 'Rev', 'Col', 'Major', 'Capt', 'Jonkheer', 'Don', 'Sir', 'Lady', 'Countess', 'Mme', 'Mlle']
train_df['Title'] = train_df['Title'].replace(rare_titles, 'Rare')
title_mapping = {'Mr': 0, 'Miss': 1, 'Mrs': 2, 'Master': 3, 'Rare': 4}
train_df['Title'] = train_df['Title'].map(title_mapping).fillna(0).astype(int)

train_df = train_df.drop('Name', axis=1)
train_df = train_df.drop('Ticket', axis=1)
train_df = train_df.drop('PassengerId', axis=1)

train_df['Sex'] = train_df['Sex'].map({'male': 0, 'female': 1})
mean_age = train_df["Age"].mean()
train_df['Age'] = train_df['Age'].fillna(mean_age).round()
train_df['Cabin'] = train_df['Cabin'].notnull().astype("int")
train_df['Embarked'] = train_df['Embarked'].map({'S': 2, 'C': 1, 'Q': 0})
train_df['FamilySize'] = train_df['SibSp'] + train_df['Parch'] + 1
train_df['IsAlone'] = (train_df['FamilySize'] == 1).astype(int)

y_train = train_df['Survived'].to_numpy()
train_df = train_df.drop('Survived', axis=1)

X_train = train_df.to_numpy()

X_train_part, X_val, y_train_part, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

In [5]:
passenger_ids = test_df['PassengerId'].copy()

test_df['Title'] = test_df['Name'].str.extract(r',\s*([A-Za-z]+)\.', expand=False)
test_df['Title'] = test_df['Title'].replace(rare_titles, 'Rare')
test_df['Title'] = test_df['Title'].map(title_mapping).fillna(0).astype(int)

test_df = test_df.drop('Name', axis=1)
test_df = test_df.drop('Ticket', axis=1)
test_df = test_df.drop('PassengerId', axis=1)

test_df['Sex'] = test_df['Sex'].map({'male': 0, 'female': 1})
test_df['Age'] = test_df['Age'].fillna(mean_age).round()
test_df['Cabin'] = test_df['Cabin'].notnull().astype("int")
test_df['Embarked'] = test_df['Embarked'].map({'S': 2, 'C': 1, 'Q': 0})
test_df['Embarked'] = test_df['Embarked'].fillna(2)
test_df['FamilySize'] = test_df['SibSp'] + test_df['Parch'] + 1
test_df['IsAlone'] = (test_df['FamilySize'] == 1).astype(int)

## 3. Finds the best parameters

In [6]:
param_grid = {
    'n_estimators': [100, 150, 200],
    'max_depth': [10, 12, 14],
    'min_samples_split': [4, 6, 8],
    'min_samples_leaf': [2, 3, 4]
}
base_rf = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(base_rf, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
grid_search.fit(X_train_part, y_train_part)

print(f"Best params: {grid_search.best_params_}")
print(f"Best CV accuracy: {grid_search.best_score_:.4f}")

Fitting 5 folds for each of 81 candidates, totalling 405 fits
Best params: {'max_depth': 14, 'min_samples_leaf': 2, 'min_samples_split': 8, 'n_estimators': 150}
Best CV accuracy: 0.8300


## 4. A model

### 4.1 Training a model

In [7]:
best_params = grid_search.best_params_
clf = RandomForestClassifier(**best_params, random_state=42)

clf.fit(X_train_part, y_train_part)
final_preds_val = clf.predict(X_val)
final_preds_train = clf.predict(X_train_part)

### 4.2 Use of metrics to assess quality

In [8]:
print(f"Validation accuracy: {accuracy_score(y_val, final_preds_val):.3f}")
print(f"Training accuracy: {accuracy_score(y_train_part, final_preds_train):.3f}")

precision = precision_score(y_val, final_preds_val)
recall = recall_score(y_val, final_preds_val)
print(f"precision: {precision:.3f}")
print(f"recall: {recall:.3f}")
print(f"F1-score: {f1_score(y_val, final_preds_val):.3f}")

Validation accuracy: 0.838
Training accuracy: 0.895
precision: 0.826
recall: 0.770
F1-score: 0.797


### 4.3 Retrain for all data¶

In [9]:
best_params = grid_search.best_params_
clf_final = RandomForestClassifier(**best_params, random_state=42)
clf_final.fit(X_train, y_train)

X_test = test_df.to_numpy()
final_predictions = clf.predict(X_test)

## 5. Final csv

In [ ]:
submission = pd.DataFrame({
    'PassengerId': passenger_ids,
    'Survived': final_predictions.astype(int)
})

submission.to_csv('../submissions/titanic-submission.csv', index=False)
print(submission.head())

   PassengerId  Survived
0          892         0
1          893         0
2          894         0
3          895         0
4          896         1
